# TypePro Python dataset shard 07/09

Settings required: **Internet ON**, accelerator **None/CPU**. Add account
secrets `KAGGLE_USERNAME` and `KAGGLE_KEY`. This notebook processes shard
`7` of `10` and publishes a private dataset named
`typepro-build-shard-07`.


In [ ]:
SHARD_INDEX = 7
SHARD_COUNT = 10
REPOSITORY = 'https://github.com/duyvu1105/TypePro.git'
BRANCH = 'main'
SEED = 13
TEST_PROJECTS = 100
VALIDATION_PROJECT_RATIO = 0.10
SLICE_LOG_EVERY = 50

from pathlib import Path

REPO_DIR = Path("/kaggle/working/TypePro")
WORK_DIR = Path(f"/kaggle/working/typepro_build_shard_{SHARD_INDEX:02d}")
PUBLISH_DIR = Path(f"/kaggle/working/publish_shard_{SHARD_INDEX:02d}")
print({
    "shard_index": SHARD_INDEX,
    "shard_count": SHARD_COUNT,
    "work_dir": str(WORK_DIR),
})


## Authenticate safely

Values are read from Kaggle Secrets and are never printed.


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
kaggle_username = secrets.get_secret("KAGGLE_USERNAME").strip()
kaggle_key = secrets.get_secret("KAGGLE_KEY").strip()
if not kaggle_username or not kaggle_key:
    raise RuntimeError("KAGGLE_USERNAME and KAGGLE_KEY must not be empty")
os.environ["KAGGLE_USERNAME"] = kaggle_username
os.environ["KAGGLE_KEY"] = kaggle_key
os.environ.pop("KAGGLE_API_TOKEN", None)
os.environ["PYTHONUNBUFFERED"] = "1"
print("Datasets will be owned by:", kaggle_username)


## Clone TypePro and install builder dependencies


In [ ]:
import shutil
import subprocess
import sys

def run(command, cwd=None):
    print("+", " ".join(map(str, command)), flush=True)
    subprocess.run([str(value) for value in command], cwd=cwd, check=True)

if not REPO_DIR.exists():
    run(["git", "clone", "--branch", BRANCH, "--single-branch", REPOSITORY, REPO_DIR])
else:
    print("Using existing repository:", REPO_DIR)

PIPELINE_DIR = REPO_DIR / "codet5p_type_retrieval"
run([sys.executable, "-m", "pip", "install", "-q", "-U", "-r", PIPELINE_DIR / "requirements-build.txt"])
# Force legacy-key-only authentication. Newer Kaggle CLI releases may
# prefer the notebook host account's automatic access token.
run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "kaggle==1.7.4.2"])


## Optional automatic resume

If the private shard dataset already exists, its archive is downloaded
and restored before slicing. A missing dataset simply means this is the
first run.


In [ ]:
import json
import zipfile

dataset_id = f"{os.environ['KAGGLE_USERNAME']}/typepro-build-shard-{SHARD_INDEX:02d}"
resume_dir = Path(f"/kaggle/working/resume_shard_{SHARD_INDEX:02d}")
resume_dir.mkdir(parents=True, exist_ok=True)
probe = subprocess.run(
    ["kaggle", "datasets", "files", dataset_id],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
has_work = (
    (WORK_DIR / "metadata" / "split_manifest.json").exists()
    or any((WORK_DIR / "raw_slices").glob("*.jsonl"))
    or any((WORK_DIR / "project_status").glob("*.json"))
)
if probe.returncode == 0 and not has_work:
    run(["kaggle", "datasets", "download", "-d", dataset_id, "-p", resume_dir, "--unzip"])
    markers = list(resume_dir.rglob("shard_manifest.json"))
    if not markers:
        archives = list(resume_dir.rglob("typepro_build_shard_*.zip"))
        if len(archives) != 1:
            raise RuntimeError(
                f"Expected one shard directory or archive, found markers={markers}, archives={archives}"
            )
        with zipfile.ZipFile(archives[0]) as bundle:
            bundle.extractall(resume_dir)
        markers = list(resume_dir.rglob("shard_manifest.json"))
    if len(markers) != 1:
        raise RuntimeError(f"Cannot uniquely locate restored shard: {markers}")
    source_build = markers[0].parent
    restored_manifest = json.loads(markers[0].read_text(encoding="utf-8"))
    if (
        restored_manifest.get("shard_index") != SHARD_INDEX
        or restored_manifest.get("shard_count") != SHARD_COUNT
    ):
        print(
            "Ignoring an incompatible previous shard Dataset; starting the new layout:",
            restored_manifest,
        )
    else:
        shutil.copytree(source_build, WORK_DIR, dirs_exist_ok=True)
        print("Restored previous shard state:", WORK_DIR)
else:
    print("Starting new shard or using current working state")


## Download metadata and create the deterministic project split


In [ ]:
prepare = PIPELINE_DIR / "prepare_dataset.py"
common = [
    "--typepro-root", REPO_DIR,
    "--work-dir", WORK_DIR,
    "--split-profile", "paper_project",
    "--test-projects", TEST_PROJECTS,
    "--validation-project-ratio", VALIDATION_PROJECT_RATIO,
    "--seed", SEED,
    "--preview-samples", 1,
    "--preview-max-chars", 1200,
]
run([sys.executable, "-u", prepare, "--stage", "metadata", *common])


## Clone repositories and build interprocedural slices


In [ ]:
run([
    sys.executable, "-u", prepare,
    "--stage", "slice",
    *common,
    "--shard-count", SHARD_COUNT,
    "--shard-index", SHARD_INDEX,
    "--slice-log-every", SLICE_LOG_EVERY,
    "--build-import-kb",
    "--download-missing-imports",
    "--kb-max-files-per-package", 3000,
])


## Verify that this shard attempted every assigned project


In [ ]:
import json
sys.path.insert(0, str(PIPELINE_DIR))
from prepare_dataset import project_from_row, read_json, stable_number

projects = set()
for split in ("train", "validation", "test"):
    for row in read_json(WORK_DIR / "metadata" / f"{split}.json"):
        projects.add(project_from_row(row))
selected = {
    project for project in projects
    if stable_number(project, SEED + 4) % SHARD_COUNT == SHARD_INDEX
}
statuses = []
for path in (WORK_DIR / "project_status").glob("*.json"):
    statuses.append(json.loads(path.read_text(encoding="utf-8")))
attempted = {item.get("project") for item in statuses}
missing = sorted(selected - attempted)
summary = {
    "shard_index": SHARD_INDEX,
    "shard_count": SHARD_COUNT,
    "selected_projects": len(selected),
    "attempted_projects": len(selected & attempted),
    "successful_projects": sum(item.get("project") in selected and "error" not in item for item in statuses),
    "failed_projects": sum(item.get("project") in selected and "error" in item for item in statuses),
    "exported_slices": sum(int(item.get("exported", 0)) for item in statuses if item.get("project") in selected),
    "missing_projects": missing,
}
print(json.dumps(summary, indent=2, ensure_ascii=False))
(WORK_DIR / "shard_manifest.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"
)
if missing:
    raise RuntimeError(f"Shard is incomplete: {len(missing)} projects missing")


## Package and publish this shard as a private Kaggle Dataset


In [ ]:
run([
    sys.executable, "-u", PIPELINE_DIR / "publish_shard.py",
    "--work-dir", WORK_DIR,
    "--payload-dir", PUBLISH_DIR,
    "--dataset-id", dataset_id,
    "--title", f"TypePro Python shard {SHARD_INDEX:02d} of {SHARD_COUNT}",
    "--message", f"Completed TypePro shard {SHARD_INDEX:02d} of {SHARD_COUNT}",
    "--expected-shard-index", SHARD_INDEX,
    "--expected-shard-count", SHARD_COUNT,
])
